In [1]:
import numpy as np
import torch
import cv2

from torch import nn
from torch.nn import functional as F
from typing import Union, Tuple, Sequence
from mamba_ssm import Mamba, Mamba2
from datasets.overall_dataset import MAFF_Dataset
from maff.utils.supervision import compute_supervision_coarse
from default_config import get_cfg_defaults
from maff.maff import MAFF

In [2]:
config = get_cfg_defaults()
dataset = MAFF_Dataset(config=config)

################################################################
Random seed: 668407247
################################################################


In [3]:
dataset.setup(stage="fit")

Loading megadepth dataset for validating: 100%|██████████| 5/5 [00:00<00:00, 373.40it/s]


In [4]:
train_dataloader = dataset.train_dataloader()

In [5]:
for data in train_dataloader:
    compute_supervision_coarse(data, 8)

    image0 = data["image0"].detach().cpu().numpy()[0, 0]
    image1 = data["image1"].detach().cpu().numpy()[0, 0]
    conf_matrix_gt = data["conf_matrix_gt"]

    _, pos_x, pos_y = torch.where(conf_matrix_gt==1.0)

    pos_image0_x = pos_x % 80
    pos_image0_y = pos_x // 80
    pos_image1_x = pos_y % 80
    pos_image1_y = pos_y // 80

    image0 = cv2.cvtColor(image0, cv2.COLOR_GRAY2BGR)
    image1 = cv2.cvtColor(image1, cv2.COLOR_GRAY2BGR)

    for x, y in zip(pos_image0_x, pos_image0_y):
        cv2.circle(image0, (int(x) * 8, int(y) * 8), 4, (0,0,255), 2)
        
    for x, y in zip(pos_image1_x, pos_image1_y):
        cv2.circle(image1, (int(x) * 8, int(y) * 8), 4, (0,0,255), 2)
        
    pass

ERROR:tornado.general:SEND Error: Host unreachable


In [3]:
input0 = torch.randint(low = 0, high = 256, size=(1, 1, config["IMAGE_SIZE"], config["IMAGE_SIZE"]), dtype=torch.float32).to(device)
input1 = torch.randint(low = 0, high = 256, size=(1, 1, config["IMAGE_SIZE"], config["IMAGE_SIZE"]), dtype=torch.float32).to(device)
mask0 = torch.randint(low = 0, high = 10, size=(1, int(config["IMAGE_SIZE"] / 4), int(config["IMAGE_SIZE"] / 4)), dtype=torch.float32).to(device)
mask1 = torch.randint(low = 0, high = 10, size=(1, int(config["IMAGE_SIZE"] / 4), int(config["IMAGE_SIZE"] / 4)), dtype=torch.float32).to(device)
data = {
    "image0": input0,
    "image1": input1,
    "mask0" : mask0,
    "mask1": mask1
}

In [4]:
conf_matrix = maff(data)

0: 0.0
1: 1.722001075744629
2: 1.980301856994629
3: 1.980301856994629
5: 1.846024513244629
6.0: 2.3048715591430664
6.1: 2.7560739517211914
6.2: 3.2068796157836914
6.3: 3.6572275161743164
6.4: 4.554749488830566
6.5: 5.451294898986816
6.6: 6.347840309143066
6.7: 7.244385719299316
Sequence Length: 32000
6: 7.244385719299316
7: 7.244385719299316
8: 15.227784156799316


-inf

In [11]:
sim_matrix = torch.einsum("blc,bsc->bls", x1, x2) / 0.1

In [14]:
conf_matrix = F.softmax(sim_matrix, 1) * F.softmax(sim_matrix, 2)

In [7]:
print(f":", torch.cuda.memory_allocated(torch.device("cuda:3"))/1024/1024/1024)

: 9.611451148986816


In [6]:
torch.cuda.empty_cache()

In [ ]:
from einops.einops import rearrange
for i, scale in enumerate(x1):
    # [B x C x H x W] -> [B x (HW) x C]
    x1[i]= rearrange(scale, 'b c h w -> b (h w) c')

In [ ]:
x1 = torch.concat(x1, dim=1)

In [ ]:
x1, x2 = pe(x1, x2)

In [ ]:
d_models = [64, 128, 256, 512]
max_hw = 840
scales = [0.5, 0.25, 0.125, 0.0625]
dtype = torch.float32
num_scales = len(scales)

In [ ]:
pos_x = torch.arange(max_hw, dtype=dtype)
pos_y = torch.arange(max_hw, dtype=dtype)
pos_z = torch.arange(num_scales, dtype=dtype)

In [ ]:
all_scales_pos_emb = []

In [ ]:
pos_z[0]

In [ ]:
for i in range(num_scales):
    d_pos_embeddings = int(np.ceil(d_models[i] / 3))
    inv_freq = 1.0 / (10000.0 ** (torch.arange(0, d_pos_embeddings, 2, dtype=dtype) / d_pos_embeddings))
    hw_pos_emb = torch.zeros(size=(max_hw, max_hw, 6 * inv_freq.shape[0]), dtype=dtype)                      # H, W, C
    
    temp_x = torch.einsum("i,j->ij", pos_x, inv_freq)
    temp_y = torch.einsum("i,j->ij", pos_y, inv_freq)
    temp_z = pos_z[i] * inv_freq
    
    sin_x = torch.sin(temp_x).unsqueeze(0)
    sin_y = torch.sin(temp_y).unsqueeze(1)
    sin_z = torch.sin(temp_z).unsqueeze(0).unsqueeze(0)
    cos_x = torch.cos(temp_x).unsqueeze(0)
    cos_y = torch.cos(temp_y).unsqueeze(1)
    cos_z = torch.cos(temp_z).unsqueeze(0).unsqueeze(0)
    
    hw_pos_emb[:, :, 0::6] = sin_x
    hw_pos_emb[:, :, 1::6] = cos_x
    hw_pos_emb[:, :, 2::6] = sin_y
    hw_pos_emb[:, :, 3::6] = cos_y
    hw_pos_emb[:, :, 4::6] = sin_z
    hw_pos_emb[:, :, 5::6] = cos_z

    # Crop the exceeding channels
    hw_pos_emb = hw_pos_emb[:, :, 0: d_models[i]]
    
    hw_pos_emb_rescaled = torch.nn.functional.interpolate(hw_pos_emb.swapaxes(0, -1).unsqueeze(0), scale_factor=(scales[i], scales[i])).squeeze(0).swapaxes(0, -1)
    
    all_scales_pos_emb.append(hw_pos_emb_rescaled)

In [ ]:
[print(i.shape) for i in all_scales_pos_emb]
temp_0 = all_scales_pos_emb[0]
temp_1 = all_scales_pos_emb[1]
temp_2 = all_scales_pos_emb[2]
temp_3 = all_scales_pos_emb[3]

In [ ]:
temp = torch.stack(all_scales_pos_emb).unsqueeze

In [ ]:
temp=torch.sin(temp_x)

In [ ]:
channels = 12
channels = int(np.ceil(channels / 6) * 2)
print(channels)

In [ ]:
inv_freq = 1.0 / (10000 ** (torch.arange(start=0, end=channels, step=2, dtype=torch.float32) / channels))
print(inv_freq)

In [ ]:
y_position = torch.ones(128).cumsum(0).float().unsqueeze(0)
print(y_position)

In [ ]:
x1 = torch.zeros(size=(3, 12, 512))
x2 = torch.zeros(size=(3, 15, 512))

for i in range(x1.shape[1]):
    x1[:, i, 0] = i
    
for i in range(x2.shape[1]):
    x2[:, i, 0] = i * 10

In [ ]:
concat_x = torch.concat(tensors=(x1, x2), dim=-2)
concat_x_flip = torch.flip(concat_x, dims=[-2])

fuse_x = torch.concat(tensors=(concat_x, concat_x_flip), dim=-2)

In [ ]:
fuse_y = fuse_x.clone()

In [ ]:
L = fuse_y.shape[1]
concat_y = 0.5*(fuse_y[:, 0:L//2, :] + fuse_y[:, L//2:, :].flip(dims=[1]))


In [ ]:
L1 = x1.shape[-2]
L2 = x2.shape[-2]
y1 = concat_y[:, 0:L1, :]
y2 = concat_y[:, L1:, :]

In [ ]:
class MambaEncoder(nn.Module):
    def __init__(
        self,
        layers_dim: Sequence[int],
        layers_type: Sequence[str],
        inner_expansion: float = 2,
        conv_dim: int = 4,
        device: torch.device = None,
        dtype: torch.dtype = None,
        using_mamba2: bool = False
    ):
        super(MambaEncoder, self).__init__()
        
        self.layers_dim: Sequence[int] = layers_dim
        self.layers_type: Sequence[str] = layers_type
        self.inner_expansion: float = inner_expansion
        self.conv_dim: int = conv_dim
        self.device: torch.device = device
        self.dtype: torch.dtype = dtype
        self.using_mamba2: bool = using_mamba2
        
        
        
        

In [ ]:
device = torch.device("cuda:4")
test_mamba = MambaLayer(device=device)

In [ ]:
print(test_mamba)

x = torch.zeros(size=(32, 1024, 1024)).to(device)

In [ ]:
y = test_mamba(x)

In [ ]:
print(y.shape)